# CKPT7: RNN Sequence Model (Monthly Counts)

This notebook implements a **true sequence model (RNN/GRU)** using monthly purchase counts per customer in the observation window. We train and evaluate on the same **temporal splits** used in CKPT2 for a fair comparison.


In [1]:
# Setup
import os, sys, random
import numpy as np
import pandas as pd

sys.path.append('..')

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)



In [2]:
# Install / import torch (CPU)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:
    import sys
    !{sys.executable} -m pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

print('Torch version:', torch.__version__)


Torch version: 2.11.0+cpu


In [3]:
from pathlib import Path
from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi
from src.baselines import evaluate_model


## Data + Temporal Splits (same as CKPT2)
We keep the **exact same temporal split strategy** so results are comparable with baselines.


In [4]:
file_2009 = Path('../data/Year 2009-2010.csv')
file_2010 = Path('../data/Year 2010-2011.csv')

if not (file_2009.exists() and file_2010.exists()):
    raise FileNotFoundError('Raw dataset files missing in ../data/')

raw = load_online_retail_ii(file_2009, file_2010)
clean = clean_data(raw)

train_cutoffs = ['2010-06-01','2010-09-01','2010-12-01','2011-03-01']
val_cutoff = '2011-06-01'
test_cutoff = '2011-09-01'
obs_months = 6
horizon_months = 3

train_df, val_df, test_df = create_temporal_splits_multi(
    clean, train_cutoffs, val_cutoff, test_cutoff,
    obs_months=obs_months, horizon_months=horizon_months
)

print('Train/Val/Test:', train_df.shape, val_df.shape, test_df.shape)


Loading Year 2009-2010...
  Loaded 525,461 rows
Loading Year 2010-2011...
  Loaded 541,910 rows

Total rows: 1,067,371

DATA CLEANING PIPELINE
Initial rows: 1,067,371
✓ Removed missing CustomerID: 243,007 rows removed
✓ Removed cancellations: 18,744 rows removed
✓ Removed invalid Quantity/Price: 71 rows removed
✓ Converted InvoiceDate to datetime
✓ Removed duplicates: 26,124 rows removed
Final rows: 779,425 (73.0% retained)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,878
Unique invoices: 36,969


CREATING TEMPORAL SPLITS (MULTI-CUTOFF TRAIN)

[1/3] Training Set (Multiple Cutoffs)

--- Train cutoff: 2010-06-01 ---

Creating window:
  Observation: 2009-12-01 to 2010-06-01 (6 months)
  Horizon: 2010-06-01 to 2010-09-01 (3 months)
  Observation transactions: 161,616
  Horizon transactions: 83,360
  Customers in observation: 2,703
  Final customers with features: 2,703
  Customers with target > 0: 1,344 (49.7%)

--- Train cutoff: 2010-09-01 ---

Creating windo

## Build Monthly Count Sequences
We create **monthly transaction counts** over the observation window and feed them to the RNN as a fixed-length sequence.


In [5]:
# Build monthly counts pivot for all data
monthly = clean.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
monthly_counts = monthly.groupby(['CustomerID', 'month']).size().unstack(fill_value=0)
monthly_counts.columns = pd.to_datetime(monthly_counts.columns)

seq_cols = [f'm{i}' for i in range(obs_months, 0, -1)]

def build_seq_features(df_split):
    seq_rows = []
    for _, row in df_split[['CustomerID', 'cutoff_date']].iterrows():
        cid = row['CustomerID']
        cutoff = pd.to_datetime(row['cutoff_date'])
        months = pd.date_range(cutoff - pd.DateOffset(months=obs_months), periods=obs_months, freq='MS')
        if cid in monthly_counts.index:
            counts = monthly_counts.loc[cid].reindex(months, fill_value=0).values
        else:
            counts = [0] * obs_months
        seq_rows.append(counts)
    return pd.DataFrame(seq_rows, columns=seq_cols, index=df_split.index)

X_train_seq = build_seq_features(train_df)
X_val_seq = build_seq_features(val_df)
X_test_seq = build_seq_features(test_df)

y_train = train_df['target'].values
Y_val = val_df['target'].values
y_test = test_df['target'].values

print('Sequence feature shape:', X_train_seq.shape)


Sequence feature shape: (12426, 6)


## PyTorch Dataset + DataLoader

In [6]:
from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        # shape: (seq_len, 1)
        return self.X[idx].unsqueeze(-1), self.y[idx]

train_ds = SeqDataset(X_train_seq, y_train)
val_ds = SeqDataset(X_val_seq, Y_val)
test_ds = SeqDataset(X_test_seq, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


## RNN Model (GRU)
We use a small GRU with ReLU output to keep predictions non‑negative.


In [7]:
class GRURegressor(nn.Module):
    def __init__(self, hidden_size=32, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        # x: (B, T, 1)
        out, _ = self.gru(x)
        last = out[:, -1, :]
        yhat = self.fc(last)
        return F.relu(yhat).squeeze(-1)

model = GRURegressor(hidden_size=32, num_layers=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.L1Loss()  # MAE


## Train + Validate
We track validation MAE and keep the best model (early stopping).


In [8]:
def evaluate_loader(loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in loader:
            yhat = model(xb)
            preds.append(yhat.cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return evaluate_model(trues, preds, 'GRU')

best_val = float('inf')
best_state = None
patience = 5
no_improve = 0

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        yhat = model(xb)
        loss = criterion(yhat, yb)
        loss.backward()
        optimizer.step()

    val_metrics = evaluate_loader(val_loader)
    val_mae = val_metrics['MAE']
    print(f"Epoch {epoch:02d} | Val MAE: {val_mae:.4f}")

    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping')
            break

# restore best
if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | Val MAE: 1.1784
Epoch 02 | Val MAE: 1.1784
Epoch 03 | Val MAE: 1.1784
Epoch 04 | Val MAE: 1.1784
Epoch 05 | Val MAE: 1.1784
Epoch 06 | Val MAE: 1.1784
Early stopping


## Test Evaluation + Compare to CKPT2 Baselines

In [9]:
test_metrics = evaluate_loader(test_loader)
print('GRU Test:', test_metrics)

# Compare vs CKPT2 baseline if available
baseline_path = Path('../results/ckpt2/baseline_metrics.json')
if baseline_path.exists():
    import json
    baseline = json.loads(baseline_path.read_text())
    # find best baseline MAE
    best_name, best_mae = None, float('inf')
    for name, m in baseline.items():
        if m['MAE'] < best_mae:
            best_mae = m['MAE']
            best_name = name
    print(f"Best CKPT2 baseline: {best_name} MAE={best_mae:.6f}")
    delta = test_metrics['MAE'] - best_mae
    print(f"GRU vs best baseline: {delta:+.6f}")


GRU Test: {'Model': 'GRU', 'MAE': 1.6221116781234741, 'RMSE': np.float64(3.6326618122309453)}
Best CKPT2 baseline: results_rf_test MAE=1.106082
GRU vs best baseline: +0.516030
